# Εύρεση Παραγόντων Ζήτησης με τη PROC GLMSELECT: Βηματική, LASSO, και Επικυρωμένη Επιλογή Προς τα Εμπρός


## Περίληψη

Μια ομάδα αναλυτικής λιανικής χρησιμοποιεί την **PROC GLMSELECT** για να ανακαλύψει ποιοι μοχλοί μάρκετινγκ και τιμολόγησης πραγματικά κινούν τις εβδομαδιαίες πωλήσεις μονάδων, διαχωρίζοντας τους πραγματικούς παράγοντες ζήτησης από τον θόρυβο. Η βηματική επιλογή με βαθμολόγηση SBC, το LASSO με 5-πτυχη διασταυρούμενη επικύρωση, και μια αναζήτηση προς τα εμπρός επικυρωμένη σε δείγμα αναμονής ανακτούν όλες το **ίδιο σύνολο πραγματικών παραγόντων** — δική μας τιμή, τιμή ανταγωνιστή, διαφημιστική δαπάνη, όγκος email, προσφορές, αργίες, την περιοχή Northeast, και το κανάλι Online — και καθένα από τα τέσσερα εμφυτευμένα δολώματα (`temp_f`, `noise1`, `noise2`, `noise3`) απορρίπτεται αυτόματα.

Οι μέθοδοι συμφωνούν επίσης στενά ως προς τα μεγέθη: κάθε μία εκτιμά μια επίδραση δικής τιμής κοντά στις **-28 μονάδες ανά δολάριο** και μια επίδραση τιμής ανταγωνιστή κοντά στο **+14**, τα πρόσημα υποκατάστασης που ενσωματώθηκαν στην εξίσωση παραγωγής δεδομένων. Η μόνη διαφωνία βρίσκεται στο περιθώριο — το διασταυρούμενα επικυρωμένο LASSO διατηρεί επιπλέον μια μικρή, στατιστικά αμελητέα αντίθεση `Region=Midwest` (εκτίμηση -8.3 με τυπικό σφάλμα 8.3) που τόσο η βηματική όσο και η επιλογή προς τα εμπρός απορρίπτουν. Μια λίστα παραγόντων που επιβιώνει σε τρεις διαφορετικές φιλοσοφίες επιλογής είναι πολύ πιο αξιόπιστη από μία συντονισμένη σε έναν μόνο κανόνα.


## Πηγές Δεδομένων

Όλα τα δεδομένα σε αυτό το notebook είναι **συνθετικά** και δημιουργούνται εσωτερικά (χωρίς εξωτερικά αρχεία, seed `20260531`). Μιμούνται μία σεζόν πάνελ ζήτησης κατάστημα-εβδομάδα για έναν λιανοπωλητή καταναλωτικών αγαθών.

| Σύνολο δεδομένων | Γραμμές | Κλίμακα | Βασικές μεταβλητές |
|---------|------|-------|---------------|
| `demand` | 100 | Κατάστημα-εβδομάδα | `units` (απόκριση: εβδομαδιαίες μονάδες που πωλήθηκαν)· `price`, `competitor` (δική μας & ανταγωνιστική τιμή ραφιού)· `adspend`, `email` (πίεση μέσων)· `promo`, `holiday` (σημαίες γεγονότων)· `region`, `channel` (επιδράσεις CLASS)· `temp_f`, `noise1`-`noise3` (δόλωμα/άσχετοι παράγοντες πρόβλεψης) |

Το `units` δημιουργείται από μια γνωστή γραμμική εξίσωση, ώστε το "σωστό" σύνολο παραγόντων να είναι επαληθεύσιμο· το `temp_f` και οι τρεις στήλες `noise` δεν φέρουν πραγματικό σήμα και υπάρχουν για να ελέγξουν αν κάθε μέθοδος επιλογής τα απορρίπτει.


# Εύρεση Παραγόντων Ζήτησης με τη PROC GLMSELECT

Ένας διευθυντής κατηγορίας έχει δεκάδες υποψήφιους επεξηγηματικούς παράγοντες για τις εβδομαδιαίες πωλήσεις: δική τιμή, τιμή ανταγωνιστή, διαφημιστική δαπάνη, όγκο email, προσφορές, αργίες, περιοχή καταστήματος, κανάλι πωλήσεων, ακόμη και τον καιρό. Η ρίψη όλων αυτών σε μία παλινδρόμηση προκαλεί υπερπροσαρμογή και ασταθείς συντελεστές. Η **PROC GLMSELECT** αυτοματοποιεί την αναζήτηση για ένα οικονομικό μοντέλο, υποστηρίζοντας βηματική, LASSO, ελαστικό δίκτυο, και επιλογή ελάχιστης γωνίας με ενσωματωμένη διασταυρούμενη επικύρωση και διαμέριση δείγματος αναμονής.

Σε αυτό το notebook:

1. Δημιουργούμε ένα ρεαλιστικό συνθετικό πάνελ ζήτησης κατάστημα-εβδομάδα με ένα *γνωστό* σύνολο πραγματικών παραγόντων (συν σκόπιμες μεταβλητές-δολώματα).
2. Εκτελούμε **βηματική επιλογή** με βαθμολόγηση SBC.
3. Εκτελούμε **LASSO** με 5-πτυχη διασταυρούμενη επικύρωση.
4. Εκτελούμε **επιλογή προς τα εμπρός επικυρωμένη σε δείγμα αναμονής 30%**.

Μια καλή μέθοδος επιλογής πρέπει να ανακτά τους πραγματικούς παράγοντες και να απορρίπτει τον θόρυβο — ας δούμε.


## 1. Δημιουργία ενός συνθετικού πάνελ ζήτησης

Η απόκριση `units` κατασκευάζεται από μια ρητή γραμμική εξίσωση, ώστε να γνωρίζουμε την πραγματική αλήθεια: η τιμή και η τιμή ανταγωνιστή, η διαφημιστική δαπάνη, ο όγκος email, οι σημαίες προσφοράς και αργίας, καθώς και οι επιδράσεις περιοχής και καναλιού έχουν όλες σημασία. Οι μεταβλητές `temp_f`, `noise1`, `noise2`, και `noise3` είναι καθαρά δολώματα χωρίς πραγματική σχέση με τις πωλήσεις. Ένα seed `call streaminit` κάνει τα δεδομένα αναπαραγώγιμα.


In [1]:
/* ---------------------------------------------------------------
   Synthetic store-week demand panel for a consumer-goods retailer.
   'units' follows a KNOWN equation; temp_f and noise1-3 are decoys.
   --------------------------------------------------------------- */
ΔΕΔΟΜΕΝΑ demand;
    CALL streaminit(20260531);
    LENGTH region $9 channel $8 promo $3;
    ΕΠΑΝΑΛΗΨΗ store_week = 1 ΕΩΣ 100;
        /* Region mix */
        u1 = rand('uniform');
        ΕΑΝ u1 < 0.34 ΤΟΤΕ region = 'Northeast';
        ΑΛΛΙΩΣ ΕΑΝ u1 < 0.67 ΤΟΤΕ region = 'Midwest';
        ΑΛΛΙΩΣ region = 'South';

        /* Sales channel */
        ΕΑΝ rand('uniform') < 0.45 ΤΟΤΕ channel = 'Store';
        ΑΛΛΙΩΣ channel = 'Online';

        /* Pricing and media drivers */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Event flags and an irrelevant weather decoy */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        ΕΑΝ rand('uniform') < 0.40 ΤΟΤΕ promo = 'Yes';
        ΑΛΛΙΩΣ promo = 'No';

        /* Pure noise predictors (no true effect) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Weekly units sold from a known structural equation */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        ΕΑΝ units < 0 ΤΟΤΕ units = 0;
        ΕΞΟΔΟΣ;
    ΤΕΛΟΣ;
    ΚΡΑΤΗΣΗ region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
ΕΚΤΕΛΕΣΗ;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds


## 2. Προφίλ των δεδομένων

Πριν από τη μοντελοποίηση, επιβεβαιώστε ότι η απόκριση και οι κύριοι συνεχείς υποψήφιοι βρίσκονται σε λογικές κλίμακες.


In [2]:
ΔΙΑΔΙΚΑΣΙΑ ΜΕΣΟΙ ΔΕΔΟΜΕΝΑ=demand n mean std MIN MAX maxdec=1;
    ΜΕΤΑΒΛΗΤΗ units price competitor adspend email;
    ΕΤΙΚΕΤΑ units="Μονάδες που πωλήθηκαν (εβδομαδιαίως)" price="Δική μας τιμή"
          competitor="Τιμή ανταγωνιστή" adspend="Διαφημιστική δαπάνη"
          email="Όγκος email";
    TITLE "Εβδομαδιαία ζήτηση και υποψήφιοι παράγοντες";
ΕΚΤΕΛΕΣΗ;

                                      Εβδομαδιαία ζήτηση και υποψήφιοι παράγοντες                                       

                                                  The MEANS Procedure

 Variable    Label                                                                       N        Mean     Std Dev     Minimum     Maximum
 -----------------------------------------------------------------------------------------------------------------------------------------
 units       Μονάδες που πωλήθηκαν (εβδομαδιαίως)                                      100       874.8       136.3       508.6      1162.3
 price       Δική μας τιμή                                                             100        14.0         3.4         8.0        19.9
 competitor  Τιμή ανταγωνιστή                                                          100        13.8         3.4         8.1        19.9
 adspend     Διαφημιστική δαπάνη                                                       100       990.6       726


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Βηματική επιλογή με βαθμολόγηση SBC

Η βηματική επιλογή εναλλάσσεται προσθέτοντας και αφαιρώντας επιδράσεις, εδώ κρινόμενη από το **Κριτήριο Bayes του Schwarz (SBC)** τόσο για τον έλεγχο εισόδου (`select=sbc`) όσο και για την τελική επιλογή μοντέλου (`choose=sbc`). Το SBC τιμωρεί την πολυπλοκότητα πιο αυστηρά από το AIC, ευνοώντας πιο λιτά μοντέλα.

Βασικές δηλώσεις και επιλογές:

- Η `CLASS region channel promo / param=reference` δηλώνει τους κατηγορικούς παράγοντες πρόβλεψης με κωδικοποίηση κελιού αναφοράς.
- Η `selection=stepwise(select=sbc choose=sbc)` οδηγεί την αναζήτηση.
- Η `details=summary` εκτυπώνει τη σύνοψη επιλογής βήμα-προς-βήμα· η `stb` προσθέτει τυποποιημένους συντελεστές ώστε οι επιδράσεις σε διαφορετικές κλίμακες να είναι συγκρίσιμες.
- Η `output out=demand_scored p=predicted r=residual` αποθηκεύει τις προσαρμοσμένες τιμές και τα υπόλοιπα για μελλοντική βαθμολόγηση.

Διαβάστε τη **Σύνοψη Βηματικής Επιλογής** ως το ίχνος αναζήτησης: ξεκινά από το πλήρες μοντέλο 12 επιδράσεων και *αφαιρεί* επιδράσεις μία-μία, απορρίπτοντας τα `noise1`, `noise2`, `temp_f`, την αντίθεση `Region=Midwest`, και το `noise3` με τη σειρά, επειδή κάθε αφαίρεση μειώνει το SBC. Ό,τι επιβιώνει στον πίνακα **Εκτιμήσεις Παραμέτρων** είναι το επιλεγμένο μοντέλο.


In [3]:
ΔΙΑΔΙΚΑΣΙΑ glmselect ΔΕΔΟΜΕΝΑ=demand seed=20260531;
    ΚΛΑΣΗ region channel promo / PARAM=reference;
    ΜΟΝΤΕΛΟ units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    ΕΤΙΚΕΤΑ units="Μονάδες που πωλήθηκαν (εβδομαδιαίως)" region="Περιοχή" channel="Κανάλι"
          promo="Προσφορά" price="Δική μας τιμή" competitor="Τιμή ανταγωνιστή"
          adspend="Διαφημιστική δαπάνη" email="Όγκος email"
          temp_f="Θερμοκρασία (°F)" holiday="Αργία"
          noise1="Θόρυβος 1" noise2="Θόρυβος 2" noise3="Θόρυβος 3";
    ΕΞΟΔΟΣ out=demand_scored p=predicted r=residual;
    TITLE "Βηματική επιλογή παραγόντων ζήτησης (SBC)";
ΕΚΤΕΛΕΣΗ;

                                      Εβδομαδιαία ζήτηση και υποψήφιοι παράγοντες                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Μονάδες που πωλήθηκαν (εβδομαδιαίως)


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                       Stepwise Selection Summary                                                        

    Step    Action                        Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  ----------------------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed                     Θόρυβος 1                 12    0


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO με διασταυρούμενη επικύρωση

Το LASSO συρρικνώνει τους συντελεστές προς το μηδέν, εκτελώντας ταυτόχρονα επιλογή και κανονικοποίηση. Αντί να σταματήσει σε ένα σταθερό κριτήριο, αφήνουμε τη **5-πτυχη διασταυρούμενη επικύρωση** να επιλέξει το σημείο στη διαδρομή LASSO με το καλύτερο σφάλμα εκτός δείγματος.

- Η `selection=lasso(choose=cv stop=none)` ιχνηλατεί ολόκληρη τη διαδρομή LASSO και επιλέγει το CV-βέλτιστο βήμα.
- Η `cvmethod=random(5)` αναθέτει παρατηρήσεις σε 5 τυχαίες πτυχές.

Η **Σύνοψη Επιλογής LASSO** δείχνει τη σειρά με την οποία εισέρχονται οι επιδράσεις καθώς η ποινή χαλαρώνει: πρώτα το `price`, μετά το `adspend`, το `competitor`, η περιοχή Northeast, το `email`, το `promo`, και το `holiday` — και τα επτά πραγματικά σήματα εισέρχονται πριν από οποιοδήποτε δόλωμα. Η διασταυρούμενη επικύρωση επιλέγει τότε το βήμα όπου το εκτός δείγματος PRESS είναι το χαμηλότερο, και ο πίνακας **Εκτιμήσεις Παραμέτρων** που προκύπτει διατηρεί ακριβώς τους γνήσιους παράγοντες (συν έναν μικρό όρο `Region=Midwest`) ενώ αποκλείει τα `temp_f`, `noise1`, `noise2`, και `noise3`, τα οποία εισέρχονται μόνο στο τέλος της διαδρομής.


In [4]:
ΔΙΑΔΙΚΑΣΙΑ glmselect ΔΕΔΟΜΕΝΑ=demand seed=20260531;
    ΚΛΑΣΗ region channel promo / PARAM=reference;
    ΜΟΝΤΕΛΟ units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv STOP=none)
          cvmethod=RANDOM(5);
    ΕΤΙΚΕΤΑ units="Μονάδες που πωλήθηκαν (εβδομαδιαίως)" region="Περιοχή" channel="Κανάλι"
          promo="Προσφορά" price="Δική μας τιμή" competitor="Τιμή ανταγωνιστή"
          adspend="Διαφημιστική δαπάνη" email="Όγκος email"
          temp_f="Θερμοκρασία (°F)" holiday="Αργία"
          noise1="Θόρυβος 1" noise2="Θόρυβος 2" noise3="Θόρυβος 3";
    TITLE "LASSO με 5-πτυχη διασταυρούμενη επικύρωση";
ΕΚΤΕΛΕΣΗ;

                                      Εβδομαδιαία ζήτηση και υποψήφιοι παράγοντες                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Μονάδες που πωλήθηκαν (εβδομαδιαίως)


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                                    LASSO Selection Summary                                                                    

    Step    Action                                 Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Επιλογή προς τα εμπρός επικυρωμένη σε δείγμα αναμονής

Μια τρίτη, συμπληρωματική στρατηγική: κατασκευάστε το μοντέλο με **επιλογή προς τα εμπρός** (οι επιδράσεις μόνο εισέρχονται, ποτέ δεν αποχωρούν), αλλά σταματήστε εκεί όπου η απόδοση σε ένα ανεξάρτητο δείγμα αναμονής είναι η καλύτερη — προστατεύοντας άμεσα από υπερπροσαρμογή.

- Η `partition fraction(validate=0.30)` δεσμεύει τυχαία το 30% των γραμμών για επικύρωση, αφήνοντας 70 παρατηρήσεις εκπαίδευσης και 30 επικύρωσης.
- Η `selection=forward(select=aic choose=validate)` προσθέτει επιδράσεις με βάση το AIC αλλά επιλέγει το τελικό μοντέλο με βάση το σφάλμα δείγματος επικύρωσης.

Ο πίνακας **Κλάσματα Διαμέρισης** επιβεβαιώνει τη διαίρεση 70/30. Η επιλογή προς τα εμπρός στη συνέχεια εισάγει επιδράσεις μέχρι το σφάλμα επικύρωσης να σταματήσει να βελτιώνεται· οι οκτώ επιδράσεις στον τελικό πίνακα **Εκτιμήσεις Παραμέτρων** είναι ακριβώς οι πραγματικοί παράγοντες, με τα τέσσερα δολώματα να μην γίνονται ποτέ δεκτά. Όταν τρεις μέθοδοι βασισμένες σε διαφορετικές αρχές καταλήγουν στους ίδιους παράγοντες, το μοντέλο είναι πολύ πιο πιθανό να είναι πραγματικό παρά τεχνούργημα οποιουδήποτε μεμονωμένου κανόνα επιλογής.


In [5]:
ΔΙΑΔΙΚΑΣΙΑ glmselect ΔΕΔΟΜΕΝΑ=demand seed=20260531;
    ΚΛΑΣΗ region channel promo / PARAM=reference;
    ΜΟΝΤΕΛΟ units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition FRACTION(validate=0.30);
    ΕΤΙΚΕΤΑ units="Μονάδες που πωλήθηκαν (εβδομαδιαίως)" region="Περιοχή" channel="Κανάλι"
          promo="Προσφορά" price="Δική μας τιμή" competitor="Τιμή ανταγωνιστή"
          adspend="Διαφημιστική δαπάνη" email="Όγκος email"
          temp_f="Θερμοκρασία (°F)" holiday="Αργία"
          noise1="Θόρυβος 1" noise2="Θόρυβος 2" noise3="Θόρυβος 3";
    TITLE "Επιλογή προς τα εμπρός με επικύρωση σε δείγμα αναμονής 30%";
ΕΚΤΕΛΕΣΗ;

                                      Εβδομαδιαία ζήτηση και υποψήφιοι παράγοντες                                       

The GLMSELECT Procedure


Dependent Variable: UNITS Μονάδες που πωλήθηκαν (εβδομαδιαίως)


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                                   Forward Selection Summary                                                                   

    Step    Action                                 Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC 


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Ερμηνεία των αποτελεσμάτων

Και οι τρεις στρατηγικές επιλογής ανακτούν το **ίδιο σύνολο πραγματικών παραγόντων ζήτησης** και απορρίπτουν κάθε δόλωμα. Διαβάζοντας απευθείας από τους πίνακες **Εκτιμήσεις Παραμέτρων**:

- Η **Τιμή** είναι ο κυρίαρχος παράγοντας. Ο τυποποιημένος συντελεστής της στο βηματικό μοντέλο είναι **-0.70**, κατά πολύ ο μεγαλύτερος σε μέγεθος, και η ακατέργαστη κλίση κυμαίνεται μεταξύ **-27.8** (βηματική και LASSO) και **-28.6** (προς τα εμπρός) μονάδων ανά δολάριο — σχεδόν ακριβώς το -28 με το οποίο δημιουργήθηκαν τα δεδομένα. Η **τιμή ανταγωνιστή** κινεί τη ζήτηση προς την αντίθετη κατεύθυνση περίπου **+14.4** σε όλες τις τρεις προσαρμογές, η συμπεριφορά υποκατάστασης που αναμένει ένας διευθυντής κατηγορίας.
- Η **διαφημιστική δαπάνη** (περίπου +0.062 μονάδες ανά δολάριο) και ο **όγκος email** (περίπου +1.2 μονάδες ανά αποστολή) και οι δύο αυξάνουν τις πωλήσεις, ποσοτικοποιώντας την απόκριση στα μέσα.
- Οι **προσφορές** και οι **αργίες** φέρουν τις μεγαλύτερες επιδράσεις γεγονότων: η αντίθεση `Promo=No` κυμαίνεται περίπου **-51 έως -57** σε σχέση με μια εβδομάδα με προσφορά, και η ανύψωση αργίας είναι περίπου **+66 έως +76** μονάδες. Η **περιοχή Northeast** (περίπου +49 έως +55) και το **κανάλι Online** (περίπου +24 έως +32) φέρουν θετικές βασικές επιδράσεις.
- Κρίσιμα, κάθε δόλωμα — `temp_f`, `noise1`, `noise2`, `noise3` — **απορρίπτεται** από τη βηματική και την επιλογή προς τα εμπρός και αποκλείεται από το CV-επιλεγμένο μοντέλο LASSO. Κάθε μέθοδος ανέκτησε το δομικό σήμα και αγνόησε τον θόρυβο.

Οι τρεις μέθοδοι δεν είναι πανομοιότυπες byte-προς-byte: η βηματική (SBC) και η επικυρωμένη σε δείγμα αναμονής αναζήτηση προς τα εμπρός καταλήγουν στις ίδιες οκτώ επιδράσεις, ενώ το διασταυρούμενα επικυρωμένο LASSO διατηρεί επιπλέον μια μικρή αντίθεση `Region=Midwest` (-8.3, τυπικό σφάλμα 8.3) — μια διαφορά στο επίπεδο θορύβου παρά μια ουσιαστική διαφωνία. Το ότι μια λίστα παραγόντων επιβιώνει στη βηματική SBC, στο διασταυρούμενα επικυρωμένο LASSO, και στην επικυρωμένη σε δείγμα αναμονής επιλογή προς τα εμπρός είναι το πραγματικό συμπέρασμα: ένα εύρημα ανθεκτικό σε τρεις διαφορετικές φιλοσοφίες επιλογής είναι πολύ πιο αξιόπιστο από ένα συντονισμένο σε ένα μόνο κριτήριο. Με την `OUTPUT OUT=demand_scored` να καταγράφει προσαρμοσμένες τιμές και υπόλοιπα, η ίδια ροή εργασίας επεκτείνεται φυσικά στη βαθμολόγηση του προγραμματισμένου ημερολογίου τιμών και προσφορών του επόμενου τριμήνου.
